# Practice 3 - Exercise 1: Sentiment Analysis với Hugging Face

## 1. Giới thiệu

Trong bài thực hành này, chúng ta sử dụng một mô hình phân tích cảm xúc đã được huấn luyện sẵn (pre-trained model) từ Hugging Face Hub để phân tích cảm xúc của một câu văn mẫu.

Quy trình thực hiện:

**Sentence → Tokenizer → Pre-trained Model → Sentiment**

## 2. Cài đặt / Import thư viện

Cài đặt và import các thư viện cần thiết để sử dụng mô hình phân tích cảm xúc từ Hugging Face.

Các thư viện chính:
- `transformers`: cung cấp tokenizer và mô hình pre-trained.
- `torch`: hỗ trợ xử lý tensor và thực thi mô hình.

In [1]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("PyTorch version      :", torch.__version__)
print("Transformers version :", transformers.__version__)
print("CUDA available       :", torch.cuda.is_available())

q:\Deep_Learning\UTH-Deep-Learning-nhom2\.venv-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version      : 2.13.0+cu126
Transformers version : 5.16.1
CUDA available       : True


: 

## 3. Tải mô hình phân tích cảm xúc Pre-trained

Sử dụng mô hình `distilbert-base-uncased-finetuned-sst-2-english` từ Hugging Face Hub.

Đây là mô hình DistilBERT đã được fine-tune cho bài toán phân tích cảm xúc tiếng Anh với hai nhãn:
- `NEGATIVE`: cảm xúc tiêu cực.
- `POSITIVE`: cảm xúc tích cực.

In [ ]:
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

print("Pre-trained model :", MODEL_NAME)
print("Model type        :", model.__class__.__name__)
print("Sentiment labels  :", model.config.id2label)

## 4. Tải Tokenizer

Tokenizer có nhiệm vụ chuyển văn bản đầu vào thành các token và biểu diễn chúng dưới dạng số để mô hình có thể xử lý.

Tokenizer được tải từ cùng checkpoint với mô hình để đảm bảo khả năng tương thích giữa tokenizer và pre-trained model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer :", tokenizer.__class__.__name__)
print("Model     :", MODEL_NAME)

Tokenizer : BertTokenizer
Model     : distilbert-base-uncased-finetuned-sst-2-english


## 5. Chuẩn bị câu văn mẫu

Chuẩn bị một câu tiếng Anh làm dữ liệu đầu vào cho quá trình phân tích cảm xúc.

Câu văn được chọn có nội dung thể hiện cảm xúc tích cực rõ ràng để thuận tiện cho việc kiểm tra kết quả dự đoán.

In [ ]:
sentence = "I really enjoyed this movie. It was amazing!"

print("Câu văn mẫu:")
print(sentence)

Câu văn mẫu:
I really enjoyed this movie. It was amazing!


## 6. Tokenization

Ở bước này, câu văn mẫu được đưa qua tokenizer để chuyển từ văn bản thành dữ liệu số mà mô hình có thể xử lý.

Kết quả tokenization gồm:
- `input_ids`: các ID số đại diện cho các token trong câu.
- `attention_mask`: cho mô hình biết những token nào cần được xử lý.

In [ ]:
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    padding=True,
    truncation=True
)

print("Kết quả Tokenization:")
print(inputs)

print("\nInput IDs:")
print(inputs["input_ids"])

print("\nAttention Mask:")
print(inputs["attention_mask"])

Kết quả Tokenization:
{'input_ids': tensor([[ 101, 1045, 2428, 5632, 2023, 3185, 1012, 2009, 2001, 6429,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Input IDs:
tensor([[ 101, 1045, 2428, 5632, 2023, 3185, 1012, 2009, 2001, 6429,  999,  102]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 7. Phân tích cảm xúc

Dữ liệu sau khi tokenization được đưa vào mô hình pre-trained để thực hiện phân tích cảm xúc.

Mô hình trả về `logits`, là các điểm số thô tương ứng với hai lớp cảm xúc:
- `NEGATIVE`
- `POSITIVE`

Do chỉ thực hiện dự đoán, mô hình được chuyển sang chế độ `eval()` và sử dụng `torch.no_grad()` để không tính gradient.

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

logits = outputs.logits

print("Logits:")
print(logits)

Logits:
tensor([[-4.3470,  4.6783]])


## 8. Hiển thị kết quả dự đoán

Các giá trị `logits` được chuyển thành xác suất bằng hàm `Softmax`.

Lớp có xác suất cao nhất được chọn làm kết quả dự đoán cuối cùng.

Kết quả bao gồm:
- Câu văn đầu vào.
- Nhãn cảm xúc dự đoán.
- Độ tin cậy của mô hình.

In [ ]:
probabilities = torch.softmax(logits, dim=1)

predicted_class_id = torch.argmax(
    probabilities,
    dim=1
).item()

predicted_label = model.config.id2label[
    predicted_class_id
]

confidence_score = probabilities[
    0, predicted_class_id
].item()

print("Câu văn          :", sentence)
print("Cảm xúc dự đoán  :", predicted_label)
print(f"Độ tin cậy       : {confidence_score:.4f}")
print(f"Độ tin cậy (%)   : {confidence_score * 100:.2f}%")

Câu văn          : I really enjoyed this movie. It was amazing!
Cảm xúc dự đoán  : POSITIVE
Độ tin cậy       : 0.9999
Độ tin cậy (%)   : 99.99%


## 9. Giải thích kết quả

Mô hình dự đoán câu văn có cảm xúc **POSITIVE** với độ tin cậy khoảng **99.99%**.

Kết quả này phù hợp với nội dung của câu vì các từ như *enjoyed* và *amazing* thể hiện cảm xúc tích cực rõ ràng.

Quy trình phân tích được thực hiện như sau:

**Sentence → Tokenizer → Input IDs / Attention Mask → Pre-trained Model → Logits → Softmax → Sentiment**

## 10. Kết luận

Trong Exercise 1, một mô hình phân tích cảm xúc đã được huấn luyện sẵn từ Hugging Face Hub được sử dụng để phân tích cảm xúc của một câu văn mẫu.

Quy trình thực hiện:

**Sentence → Tokenizer → Pre-trained Model → Sentiment**

Kết quả cho thấy câu:

`"I really enjoyed this movie. It was amazing!"`

được dự đoán là **POSITIVE** với độ tin cậy khoảng **99.99%**.

Qua bài thực hành, đã hoàn thành đầy đủ các yêu cầu:
- Cài đặt và import thư viện `transformers`.
- Sử dụng mô hình sentiment analysis pre-trained.
- Tải tokenizer tương ứng.
- Chuẩn bị câu văn mẫu.
- Tokenize câu văn.
- Thực hiện sentiment analysis.
- Hiển thị kết quả dự đoán.
- Giải thích kết quả.